In [8]:
import numpy as np
from scipy.ndimage import zoom

def read_ascii_grid(filepath):
    with open(filepath, 'r') as f:
        header = {}
        data_start = 0
        for _ in range(6):  # Try to read up to 6 lines
            line = f.readline()
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            key = parts[0].lower()
            value = parts[1]
            try:
                # Handle NaN and float/integer cases
                if value.lower() == 'nan':
                    header[key] = np.nan
                else:
                    header[key] = float(value) if '.' in value or 'e' in value.lower() else int(value)
            except ValueError:
                header[key] = value  # fallback for unknown cases
            data_start += 1

        if 'nodata_value' not in header:
            header['nodata_value'] = -9999

        data = np.loadtxt(filepath, skiprows=data_start)
    return header, data


def write_ascii_grid(filepath, header, data):
    with open(filepath, 'w') as f:
        f.write(f"NCOLS {header['ncols']}\n")
        f.write(f"NROWS {header['nrows']}\n")
        f.write(f"XLLCORNER {header['xllcorner']}\n")
        f.write(f"YLLCORNER {header['yllcorner']}\n")
        f.write(f"CELLSIZE {header['cellsize']}\n")
        f.write(f"NODATA_VALUE {header['nodata_value']}\n")
        for row in data:
            f.write(" ".join(map(str, row)) + "\n")

# --- File paths (replace with your actual file paths) ---
file1_path = "test_five.asc"  # High-res values
file2_path = "tick_bite_heatmap.asc"  # Structure to keep
output_path = "output_five.asc"

# --- Read both files ---
header1, data1 = read_ascii_grid(file1_path)
header2, _ = read_ascii_grid(file2_path)

# --- Resample data1 to match shape of file2 ---
zoom_factors = (
    header2['nrows'] / data1.shape[0],
    header2['ncols'] / data1.shape[1]
)
resampled_data = zoom(data1, zoom=zoom_factors, order=1)  # bilinear interpolation
resampled_data = np.round(resampled_data, decimals=2)  # optional rounding

# Replace NaNs with NODATA value if needed
resampled_data[np.isnan(resampled_data)] = header2.get('nodata_value', -9999)

# --- Write new ASCII file with file2's header and resampled values ---
write_ascii_grid(output_path, header2, resampled_data)
